# Aim of Notebook 

This notebook constructs a structured corpus of English-language BTS reader-insert fanfiction using the GOLEM SPARQL endpoint. The goal is to retrieve, and organise fanfiction metadata (including fic summary, content ratings and tags, relationships, and engagement metrics such as kudos) into a corpora.  The resulting corpora will further support comparative analysis of tropes and character relationships in fanfiction in relation to BTS character.ai chatbot.

## 1. Setting Up Notebook Environment and Imports 

This section sets up access to project-level scripts and modules by adjusting the Python path and imports required libraries. It also initializes a custom SPARQLClient to standardize queries to the GOLEM endpoint throughout the notebook.

In [1]:
# Notebook location:
#     notebooks/02_bts_fics_corpus

# Project structure:
#     DH_Thesis/
#     ├── notebooks/
#     ├── scripts/
#     └── data/
#     └── reports/


# system utilities for module path handling and OS operations

import sys
import os
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent  
DATA_DIR = PROJECT_ROOT / "data"
FICS_DIR = DATA_DIR / "bts_fics"
ALT_DIR = FICS_DIR / "alternate-corpus"

#structured data handling of SPARQL results in dataframes

import pandas as pd

# tqdm for progress bars during batch SPARQL queries

from tqdm import tqdm

# defaultdict is a dictionary subclass that automatically creates a default value
# for missing keys, so you don’t need to check if a key exists before using it.
from collections import defaultdict


#sys path moving to Project_Root
sys.path.append(str(PROJECT_ROOT))

# Import custom SPARQL client module from scripts 
#
# SPARQLClient is a utility class that
# manages connection to the GOLEM SPARQL
# endpoint and simplifies query execution.

from scripts.sparql_client import SPARQLClient


# initialize reusable SPARQL connection object
client = SPARQLClient()

D:\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 2. Data Retrieval from the GOLEM Database Using SPARQL Endpoint



### 2.1 Tags for Retreiving BTS reader-insert Fanfiction  

This section defines the relationship tags as practiced by fanfiction authors and readers on Archive of Our Own. These tags are used to identify **reader-insert BTS fanfiction for each member of the KPop boyband**. The tag dictionary serves as the input configuration for the corpus construction so that the retrieval process of specifically reader-insert fanfiction is repeated consistently across all seven BTS members.

In [2]:
readerinsertfic_tags = {
    "jungkook": ["Jeon Jungkook/Reader", "Jeon Jungkook/You"],
    "taehyung": ["Kim Taehyung | V/Reader","Kim Taehyung | V/You"],
    "jimin": ["Park Jimin (BTS)/Reader","Park Jimin (BTS)/You"],
    "namjoon": ["Kim Namjoon | RM/Reader","Kim Namjoon | RM/You"],
    "hoseok": ["Jung Hoseok | J-Hope/Reader","Jung Hoseok | J-Hope/You"],
    "yoongi": ["Min Yoongi | Suga/Reader","Min Yoongi | Suga/You"],
    "seokjin": ["Kim Seokjin | Jin/Reader", "Kim Seokjin | Jin/You"]
}

### 2.2 Dynamic Query Builder for English BTS Reader-insert Fanfiction By Popularity (Kudos)

The `build_query()` function generates a member-specific SPARQL query from a list of relationship tags. The query retrieves all **English-language BTS fanfiction** matching the selected reader-insert relationship tags and returns a unique set of works **ranked by kudos** (popularity metric in fanfiction archive "Archive of Our Own". 

Since the GOLEM database may contain multiple observations of the same fanfiction collected over time, the query uses MAX() aggregation to retain the **highest available kudos count** and ensure a single record per fanfiction story.

In [3]:
def build_query(tags):
    if not tags:
        raise ValueError("Error: Tag list is empty")
    tag_filter = " || ".join(
        f'STRSTARTS(?rel, "{t}")' for t in tags
    )

    return f"""
    PREFIX golem: <https://golemlab.eu/graph/>

    SELECT ?s (MAX(xsd:integer(?kudos)) AS ?kudos)
    WHERE {{
      ?s golem:fandom "Bangtan Boys" .
      ?s golem:language "English" .
      ?s golem:socialRelationships ?rel .
      ?s golem:numberOfKudos ?kudos .

      FILTER({tag_filter})
      FILTER(REGEX(STR(?kudos), "^[0-9]+$"))
    }}
    GROUP BY ?s
    ORDER BY DESC(?kudos)
    """

### 2.3 Retrieving Fanfiction Identifiers and Kudos into a DataFrame

The `get_fic_ids()` function executes the aforementioned SPARQL query and converts the results into a Pandas DataFrame. Each row represents a **unique fanfic URL together with its associated kudos count**. 

This forms the base corpus for subsequent metadata retrieval of each fanfiction. 

In [4]:
def get_fic_ids(tags):
    query = build_query(tags)
    results = client.select(query)

    return pd.DataFrame([
        {
            "s": r["s"]["value"],
            "kudos": int(r["kudos"]["value"])
        }
        for r in results
    ])

### 2.4 Chunk List Function 

Before moving to corpus construction, the `chunk_list()` function is defined to split large entity lists into smaller batches to reduce query load on the GOLEM SPARQL endpoint hosted on a Virtuoso RDF store. 

Following established practices in SPARQL pagination and linked data retrieval systems for handling large result sets incrementally rather than through a single large query, this batching approach ensures stable query performance and helps avoid HTTP 503/429 errors by keeping each request within manageable rate and connection limits  ([Patrick van Kleef 2018](https://medium.com/virtuoso-blog/dbpedia-usage-report-as-of-2018-01-01-8cae1b81ca71)).


In [5]:
# Splits a list into smaller batches of fixed size
# Used to avoid overloading the SPARQL endpoint when sending large queries

def chunk_list(lst, size=50):
    for i in range(0, len(lst), size):
        # yield to create a generator that returns one chunk at a time
        # instead of storing all chunks in memory at once.
        yield lst[i:i+size]

### 2.5 Query for Fetching Fanfiction Metadata in Batches 

The `fetch_metadata()` function retrieves descriptive metadata for each batch of fanfics using a SPARQL VALUES clause. For each work, metadata such as: - title, 
- summary text of the fanfiction,
- publication date,
- word count,
- content-rating (level of sexuality/violence),
- publication status (in-progress/completed work),
- keywords (user-defined content tags),
- relationship tags (social- romantic, platonic or sexual relationships between characters) are collected. 





In [6]:
def fetch_metadata(batch):
    values = " ".join(f"<{x['s']}>" for x in batch)

    query = f"""
    PREFIX golem: <https://golemlab.eu/graph/>

    SELECT ?s ?title ?summary ?datePublished ?words ?rating ?romanticCategory ?publicationStatus ?keyword ?rel WHERE {{
      VALUES ?s {{ {values} }}

      ?s golem:title ?title .
      ?s golem:summary ?summary .
      ?s golem:datePublished ?datePublished .
      ?s golem:numberOfWords ?words .
      ?s golem:rating ?rating .
      ?s golem:romanticCategory ?romanticCategory .
      ?s golem:publicationStatus ?publicationStatus .

      # Retrieve keywords and relationships when available;
      # keep the fanfic in the results if these fields are missing.

      OPTIONAL {{ ?s golem:keyword ?keyword }}
      OPTIONAL {{ ?s golem:socialRelationships ?rel }}
    }}
    """

    return client.select(query)

## 3. Corpus Construction 

### 3.1 Constructing the Corpus for Each BTS Member 

The `build_corpus()` function orchestrates the full retrieval workflow. It first obtains the list of fanfic identifiers, retrieves metadata in batches, converts the results into a DataFrame, and aggregates repeated metadata fields generated by the RDF structure of the dataset.

Since a single fanfiction contains multiple keywords and relationship tags, metadata retrieval returns multiple rows per work. These rows are collapsed into a single record per fanfic by grouping on the unique fanfic URI and concatenating all unique keyword and relationship values.

The cleaned metadata table is merged with the original fanfic identifier table, preserving the corpus definition established by the initial kudos-ranked query while enriching each work with its associated metadata.

In [7]:
def build_corpus(tags):
    df_ids = get_fic_ids(tags)

    all_data = []

    for batch in tqdm(list(chunk_list(df_ids.to_dict("records"), 50))):
        res = fetch_metadata(batch)

        for r in res:
            all_data.append({
                "s": r["s"]["value"],
                "title": r.get("title", {}).get("value"),
                "summary": r.get("summary", {}).get("value"),
                "datePublished": r.get("datePublished", {}).get("value"),
                "words": r.get("words", {}).get("value"),
                "rating": r.get("rating", {}).get("value"),
                "romanticCategory": r.get("romanticCategory", {}).get("value"),
                "publicationStatus": r.get("publicationStatus", {}).get("value"),
                "keyword": r.get("keyword", {}).get("value"),
                "relationship": r.get("rel", {}).get("value"),
            })

    df = pd.DataFrame(all_data)

    #grouping multiple rows of keywords/relationship tags for one work together
    df_clean = df.groupby("s").agg({
        "title": "first",
        "summary": "first",
        "datePublished": "first",
        "words": "first",
        "rating": "first",
        "romanticCategory": "first",
        "publicationStatus": "first",
    
        "keyword": lambda x: " | ".join(
            sorted(set(str(i) for i in x if pd.notna(i)))
        ),
    
        "relationship": lambda x: " | ".join(
            sorted(set(str(i) for i in x if pd.notna(i)))
        )
    }).reset_index()

    #merge metadata with fanfiction ID and sorted kudos 
    return df_ids.merge(df_clean, on="s", how="left")

### 3.2 Intializing Function for all BTS Band Members 

The retrieval pipeline is executed iteratively for all seven BTS members. For each member, a separate corpus is constructed using the predefined reader-insert relationship tags and stored within a corpus dictionary for later export and analysis.

In [8]:
corpora = {}

for member, tags in readerinsertfic_tags.items():
    print(f"Building corpus for {member}...")
    df = build_corpus(tags)
    corpora[member] = df

Building corpus for jungkook...


100%|██████████████████████████████████████████████████████████████████████████████████| 57/57 [00:42<00:00,  1.33it/s]


Building corpus for taehyung...


100%|██████████████████████████████████████████████████████████████████████████████████| 23/23 [00:11<00:00,  1.97it/s]


Building corpus for jimin...


...

100%|██████████████████████████████████████████████████████████████████████████████████| 22/22 [00:11<00:00,  1.92it/s]


Building corpus for yoongi...


100%|██████████████████████████████████████████████████████████████████████████████████| 27/27 [00:19<00:00,  1.37it/s]


Building corpus for seokjin...


100%|██████████████████████████████████████████████████████████████████████████████████| 15/15 [00:05<00:00,  2.76it/s]


### 3.3 Corpora Size 

In [9]:
#check if all members data is retrieved 
print(corpora.keys()) 

dict_keys(['jungkook', 'taehyung', 'jimin', 'namjoon', 'hoseok', 'yoongi', 'seokjin'])


In [10]:
#count of fic per member 
for member, df in corpora.items():
    print("reader-insert fanfiction featuring", member, "are", len(df))

reader-insert fanfiction featuring jungkook are 2833
reader-insert fanfiction featuring taehyung are 1110
reader-insert fanfiction featuring jimin are 1090
reader-insert fanfiction featuring namjoon are 921
reader-insert fanfiction featuring hoseok are 1063
reader-insert fanfiction featuring yoongi are 1305
reader-insert fanfiction featuring seokjin are 729


## 4. Saving BTS Reader-Insert Fanfiction Corpora 

The `save_bts_fics()` function exports the completed fanfiction corpora to CSV format for subsequent analysis. Each member-specific corpus is saved separately and sorted by kudos in descending order to preserve the ranking established during corpus construction. A member column is added to retain information about the corpus source.

The function then concatenates all member corpora into a single combined dataset **(bts_all_fics.csv)**. This is likewise sorted by kudos and saved to the project data directory. The combined corpus facilitates cross-member analyses while preserving member-level information through the added metadata column.

In [13]:
def save_bts_fics(corpora, save_dir=FICS_DIR):
    os.makedirs(save_dir, exist_ok=True)

    all_dfs = []

    for member, df in corpora.items():
        df = df.copy()
        df["member"] = member

        df = df.sort_values("kudos", ascending=False)
        
        path = save_dir / f"{member}_fics.csv"
        df.to_csv(path, index=False)

        print(f"Saved {member} → {path}")

        all_dfs.append(df)

    combined = pd.concat(all_dfs, ignore_index=True)
    combined = combined.sort_values("kudos", ascending=False)

    combined_path = save_dir / "bts_all_fics.csv"
    combined.to_csv(combined_path, index=False)

    print(f"\nSaved combined dataset → {combined_path}")

    return combined

In [14]:
combined_df = save_bts_fics(corpora)

Saved jungkook → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\jungkook_fics.csv
Saved taehyung → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\taehyung_fics.csv
Saved jimin → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\jimin_fics.csv
Saved namjoon → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\namjoon_fics.csv
Saved hoseok → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\hoseok_fics.csv
Saved yoongi → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\yoongi_fics.csv
Saved seokjin → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\seokjin_fics.csv

Saved combined dataset → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\bts_all_fics.csv


## 5. Alternate Corpus Construction: Exclusive Single-Member Reader-Insert Fanfiction Corpus 

Sometimes a single fanfiction includes multiple BTS members paired with a single reader-character. This usually happens for two reasons: firstly, the work is a collection of short texts such as drabbles (very short fanfiction pieces, often around 100 words), one-shots (complete standalone stories written as a single narrative without sequels or chapters), or standalone entries within a series; secondly, the plot is built around a polyamorous relationship (polyships), often involving all BTS members (commonly referred to as OT7 in the BTS fandom), often sharing a romantic and/or sexual relationship with the reader.

This alternative corpus is constructed to separate strictly single-member reader-insert fanfiction narratives from multi-member x reader or polyship fanfictions.

At the corpus construction stage, the working hypothesis is that this filtered corpus may enable a clearer comparison with character.ai bots, since fanbots are designed to typically represent a single BTS member interacting with a user in isolation, rather than within multi-member or polyamorous relational structures.

### 5.1 Building Alternative Corpus

In [15]:

# Copy dataframe safely
df_check = combined_df.copy()

#  Map AO3 relationship tags → BTS members
tag_to_member = {}

for member, tags in readerinsertfic_tags.items():
    for t in tags:
        tag_to_member[t] = member


# Function to extract all matched members per fic
# (based on substring matching in relationship field)

def get_members(rel):
    if not isinstance(rel, str):
        return set()
    
    members = set()
    
    for tag, member in tag_to_member.items():
        if tag in rel:
            members.add(member)
    
    return members


# Apply membership detection to each fic
df_check["members_found"] = df_check["relationship"].apply(get_members)


# Filter: keep ONLY fics that belong to exactly ONE member
single_member_fics = df_check[df_check["members_found"].apply(lambda x: len(x) == 1)].copy()




In [16]:
# TOTAL corpus sizes (from SPARQL pipeline)
total_counts = {
    member: len(df)
    for member, df in corpora.items()
}


# reader x single-member counts

single_member_fics = df_check[df_check["members_found"].apply(lambda x: len(x) == 1)].copy()
single_member_fics["member"] = single_member_fics["members_found"].apply(lambda x: list(x)[0])

exclusive_counts = single_member_fics.groupby("member")["s"].nunique().to_dict()


#comparison
all_members = sorted(set(total_counts) | set(exclusive_counts))

for m in all_members:
    total = total_counts.get(m, 0)
    exclusive = exclusive_counts.get(m, 0)
    print(f"{m}: total={total} | exclusive={exclusive}")

hoseok: total=1063 | exclusive=878
jimin: total=1090 | exclusive=1069
jungkook: total=2833 | exclusive=1755
namjoon: total=921 | exclusive=797
seokjin: total=729 | exclusive=664
taehyung: total=1110 | exclusive=1021
yoongi: total=1305 | exclusive=1230


### 5.2 Saving Alternate Corpus 

In [22]:

# Save exclusive single-member corpus

save_dir = ALT_DIR
os.makedirs(save_dir, exist_ok=True)

# save per member files
for member in single_member_fics["member"].unique():
    member_df = single_member_fics[single_member_fics["member"] == member].copy()

    path = save_dir / f"{member}_exclusive_fics.csv"
    member_df.to_csv(path, index=False)

    print(f"Saved {member} → {path}")


# Save combined exclusive corpus

combined_exclusive_path = save_dir / "bts_exclusive_all_fics.csv"
single_member_fics.to_csv(combined_exclusive_path, index=False)

print(f"\nSaved combined exclusive corpus → {combined_exclusive_path}")

Saved yoongi → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\alternate-corpus\yoongi_exclusive_fics.csv
Saved jungkook → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\alternate-corpus\jungkook_exclusive_fics.csv
Saved jimin → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\alternate-corpus\jimin_exclusive_fics.csv
Saved hoseok → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\alternate-corpus\hoseok_exclusive_fics.csv
Saved taehyung → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\alternate-corpus\taehyung_exclusive_fics.csv
Saved namjoon → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\alternate-corpus\namjoon_exclusive_fics.csv
Saved seokjin → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\alternate-corpus\seokjin_exclusive_fics.csv

Saved combined exclusive corpus → E:\Jupyter_Programs\DH_Thesis\data\bts_fics\alternate-corpus\bts_exclusive_all_fics.csv
